In [161]:
import pandas as pd

DATA = "data/"  # change to wherever your CSVs are
exams    = pd.read_csv(DATA + "exams.csv")
periods  = pd.read_csv(DATA + "periods.csv")
rooms    = pd.read_csv(DATA + "rooms.csv")
avail    = pd.read_csv(DATA + "room_availability.csv")
draft    = pd.read_csv(DATA + "draft_schedule.csv")
external = pd.read_csv(DATA + "external_exams.csv")
enroll   = pd.read_csv(DATA + "enrollments.csv")

for name, df in [("exams", exams), ("periods", periods), ("rooms", rooms),
                 ("avail", avail), ("draft", draft), ("external", external), ("enroll", enroll)]:
    print(name, df.shape)

exams (24, 4)
periods (26, 5)
rooms (4, 2)
avail (32, 2)
draft (24, 3)
external (7, 3)
enroll (1650, 2)


In [162]:
avail

,room_id,period_id
0,Klaus-1447,P03
1,Klaus-1447,P05
2,Klaus-1447,P09
3,Klaus-1447,P14
4,Klaus-1447,P17
5,Klaus-1447,P21
6,Klaus-1447,P22
7,Klaus-1447,P23
8,Klaus-1456,P03
9,Klaus-1456,P05


In [163]:
reg_count = 0
read_count = 0
conflict_count = 0
for p in periods["period_type"]:
    if p == "regular":
        reg_count += 1
    elif p == "reading":
        read_count += 1
    else:
        conflict_count += 1

In [164]:
print(reg_count, read_count, conflict_count)

21 3 2


In [165]:
periods["period_type"].value_counts()

period_type
regular     21
reading      3
conflict     2
Name: count, dtype: int64

In [166]:
avail["period_id"].nunique()

8

In [167]:
avail["period_id"].nunique()
avail.groupby("room_id").size()

room_id
Klaus-1443    8
Klaus-1447    8
Klaus-1456    8
Klaus-2443    8
dtype: int64

In [168]:
isye = enroll[~enroll["exam_id"].str.contains("COMMON")]
sizes = isye["exam_id"].value_counts()
sizes

exam_id
ISYE4133-A    66
ISYE4134-A    66
ISYE3104-A    65
ISYE3133-B    65
ISYE3044-A    64
ISYE3103-A    64
ISYE4112-A    64
ISYE2027-B    64
ISYE3104-B    63
ISYE3232-B    61
ISYE3103-B    60
ISYE2027-A    60
ISYE2028-A    59
ISYE4111-A    59
ISYE4034-A    59
ISYE3039-A    59
ISYE4031-A    58
ISYE3133-A    56
ISYE3770-A    56
ISYE3133-C    55
ISYE3232-A    55
ISYE3044-B    54
ISYE2028-B    54
ISYE3030-A    54
Name: count, dtype: int64

In [169]:
external.merge(periods, on="period_id")

,exam_id,course_code,period_id,date,start_time,end_time,period_type
0,ECE3710-COMMON,ECE 3710,P04,2026-12-10,18:00,20:50,regular
1,MATH1551-COMMON,MATH 1551,P04,2026-12-10,18:00,20:50,regular
2,PHYS2211-COMMON,PHYS 2211,P08,2026-12-11,18:00,20:50,regular
3,PHYS2212-COMMON,PHYS 2212,P12,2026-12-14,18:00,20:50,regular
4,MATH1553-COMMON,MATH 1553,P16,2026-12-15,18:00,20:50,regular
5,MATH1554-COMMON,MATH 1554,P16,2026-12-15,18:00,20:50,regular
6,MATH1552-COMMON,MATH 1552,P20,2026-12-16,18:00,20:50,regular


In [170]:
common = enroll[enroll["exam_id"].str.contains("COMMON")]
common["student_id"].nunique()

210

In [171]:
# usable ISyE periods with their day and time
usable = periods[periods["period_id"].isin(avail["period_id"].unique())]

# external exams with their day and time
ext = external.merge(periods, on="period_id")

# for each day: the external period(s) and the ISyE period(s)
table = (
    periods[["date"]].drop_duplicates()
    .merge(ext.groupby("date")["period_id"].apply(list).rename("external"),
           on="date", how="left")
    .merge(usable.groupby("date")["period_id"].apply(list).rename("isye"),
           on="date", how="left")
)
table

,date,external,isye
0,2026-12-10,"[P04, P04]",[P03]
1,2026-12-11,[P08],[P05]
2,2026-12-14,[P12],[P09]
3,2026-12-15,"[P16, P16]",[P14]
4,2026-12-16,[P20],[P17]
5,2026-12-17,NaN,"[P21, P22, P23]"
6,2026-12-18,NaN,NaN


In [172]:
periods

,period_id,date,start_time,end_time,period_type
0,P01,2026-12-10,08:00,10:50,reading
1,P02,2026-12-10,11:20,14:10,reading
2,P03,2026-12-10,14:40,17:30,regular
3,P04,2026-12-10,18:00,20:50,regular
4,P05,2026-12-11,08:00,10:50,regular
5,P06,2026-12-11,11:20,14:10,regular
6,P07,2026-12-11,14:40,17:30,regular
7,P08,2026-12-11,18:00,20:50,regular
8,P09,2026-12-14,08:00,10:50,regular
9,P10,2026-12-14,11:20,14:10,regular


In [173]:
E = exams["exam_id"].tolist()
P = sorted(avail["period_id"].unique())

include_room4 = False   # flip to True for the 4-room run

if not include_room4:
    rooms = rooms[rooms["room_id"] != "Klaus-2443"]
    avail = avail[avail["room_id"] != "Klaus-2443"]

R = rooms["room_id"].tolist()
cap = dict(zip(rooms["room_id"], rooms["capacity"]))

print(E)
print(P)
print(R)
print(cap)

['ISYE2027-A', 'ISYE2027-B', 'ISYE2028-A', 'ISYE2028-B', 'ISYE3030-A', 'ISYE3039-A', 'ISYE3044-A', 'ISYE3044-B', 'ISYE3103-A', 'ISYE3103-B', 'ISYE3104-A', 'ISYE3104-B', 'ISYE3133-A', 'ISYE3133-B', 'ISYE3133-C', 'ISYE3232-A', 'ISYE3232-B', 'ISYE3770-A', 'ISYE4031-A', 'ISYE4034-A', 'ISYE4111-A', 'ISYE4112-A', 'ISYE4133-A', 'ISYE4134-A']
['P03', 'P05', 'P09', 'P14', 'P17', 'P21', 'P22', 'P23']
['Klaus-1447', 'Klaus-1456', 'Klaus-1443']
{'Klaus-1447': 75, 'Klaus-1456': 100, 'Klaus-1443': 200}


In [174]:
size = sizes.to_dict()
print(size["ISYE4133-A"])
print(len(size))

66
24


In [175]:
course = dict(zip(exams["exam_id"], exams["course_code"]))
course

{'ISYE2027-A': 'ISYE 2027',
 'ISYE2027-B': 'ISYE 2027',
 'ISYE2028-A': 'ISYE 2028',
 'ISYE2028-B': 'ISYE 2028',
 'ISYE3030-A': 'ISYE 3030',
 'ISYE3039-A': 'ISYE 3039',
 'ISYE3044-A': 'ISYE 3044',
 'ISYE3044-B': 'ISYE 3044',
 'ISYE3103-A': 'ISYE 3103',
 'ISYE3103-B': 'ISYE 3103',
 'ISYE3104-A': 'ISYE 3104',
 'ISYE3104-B': 'ISYE 3104',
 'ISYE3133-A': 'ISYE 3133',
 'ISYE3133-B': 'ISYE 3133',
 'ISYE3133-C': 'ISYE 3133',
 'ISYE3232-A': 'ISYE 3232',
 'ISYE3232-B': 'ISYE 3232',
 'ISYE3770-A': 'ISYE 3770',
 'ISYE4031-A': 'ISYE 4031',
 'ISYE4034-A': 'ISYE 4034',
 'ISYE4111-A': 'ISYE 4111',
 'ISYE4112-A': 'ISYE 4112',
 'ISYE4133-A': 'ISYE 4133',
 'ISYE4134-A': 'ISYE 4134'}

In [176]:
# 4. allowed (room, period) pairs, as a set of tuples
A = set(zip(avail["room_id"], avail["period_id"]))
A

{('Klaus-1443', 'P03'),
 ('Klaus-1443', 'P05'),
 ('Klaus-1443', 'P09'),
 ('Klaus-1443', 'P14'),
 ('Klaus-1443', 'P17'),
 ('Klaus-1443', 'P21'),
 ('Klaus-1443', 'P22'),
 ('Klaus-1443', 'P23'),
 ('Klaus-1447', 'P03'),
 ('Klaus-1447', 'P05'),
 ('Klaus-1447', 'P09'),
 ('Klaus-1447', 'P14'),
 ('Klaus-1447', 'P17'),
 ('Klaus-1447', 'P21'),
 ('Klaus-1447', 'P22'),
 ('Klaus-1447', 'P23'),
 ('Klaus-1456', 'P03'),
 ('Klaus-1456', 'P05'),
 ('Klaus-1456', 'P09'),
 ('Klaus-1456', 'P14'),
 ('Klaus-1456', 'P17'),
 ('Klaus-1456', 'P21'),
 ('Klaus-1456', 'P22'),
 ('Klaus-1456', 'P23')}

In [177]:
# 5. the draft: exam -> period, exam -> room
draft_period = dict(zip(draft["exam_id"], draft["period_id"]))
draft_room   = dict(zip(draft["exam_id"], draft["room_id"]))

In [178]:
# 6. period timing
day   = dict(zip(periods["period_id"], periods["date"]))
start = dict(zip(periods["period_id"], periods["start_time"]))

In [179]:
# 7. exam -> set of students   (GROUP BY exam_id, collect students)
isye = enroll[enroll["exam_id"].isin(E)]
S_e = isye.groupby("exam_id")["student_id"].apply(set).to_dict()

In [180]:
# 8. student -> set of ISyE exams   (same thing, flipped)
E_s = isye.groupby("student_id")["exam_id"].apply(set).to_dict()

In [181]:
# 9. student -> set of external periods
ext = enroll[~enroll["exam_id"].isin(E)].merge(external, on="exam_id")
ext_s = ext.groupby("student_id")["period_id"].apply(set).to_dict()

In [182]:
print(course["ISYE3133-B"])         # ISYE 3133
print(len(A))                       # 24 with 3 rooms, 32 with 4
print(draft_period["ISYE3133-A"])   # P14
print(day["P21"], start["P21"])     # 2026-12-17 08:00
print(len(S_e["ISYE4133-A"]))       # 66
print(E_s["S001"])                  # 4 ISyE exams
print(len(ext_s))                   # 210
print(ext_s["S001"])                # {'P16'}

ISYE 3133
24
P14
2026-12-17 08:00
66
{'ISYE4031-A', 'ISYE3044-B', 'ISYE3104-A', 'ISYE2028-A'}
210
{'P16'}
